# Serve Muse Glimmer 30B for Pocket AOI

Runs the open-weight model on Kaggle's free T4 GPUs, and gives your computer a private link to it.

**Before running:** in the right-hand panel, set the accelerator to **GPU T4 x2** and turn **Internet on**. Both need a phone-verified Kaggle account.

Then choose **Run > Run all**, and keep this tab open. The last cell runs until you stop the session, and that is expected.

The key in step 5's output is private. Do not make this notebook public, and do not share that output.

## 1. Check the machine

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!df -h /kaggle/temp
!which nvcc


## 2. Build the server (about 15-25 minutes, and it shows nothing until it finishes)

In [ ]:
import os
import subprocess

# Kaggle provides the GPU driver only as libcuda.so.1, but the build looks for the name libcuda.so.
link = "/usr/local/cuda/lib64/libcuda.so"
found = subprocess.run(["find", "/usr", "/opt", "-name", "libcuda.so.1"], capture_output=True, text=True).stdout.split()
print("GPU driver library:", found[0] if found else "NOT FOUND")
if found and not os.path.lexists(link):
    os.symlink(found[0], link)

!cd /kaggle/temp && ([ -d llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp)
!rm -rf /kaggle/temp/llama.cpp/build
!cmake /kaggle/temp/llama.cpp -B /kaggle/temp/llama.cpp/build -DGGML_CUDA=ON -DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF > /kaggle/temp/cmake.log 2>&1 || grep -A8 "CMake Error" /kaggle/temp/cmake.log
!cmake --build /kaggle/temp/llama.cpp/build --config Release -j4 --target llama-server > /kaggle/temp/build.log 2>&1 || grep -m20 -A4 "error" /kaggle/temp/build.log
!/kaggle/temp/llama.cpp/build/bin/llama-server --version


## 3. Download the model onto Kaggle's disk (about 18 GB, nothing goes to your computer)

In [ ]:
!pip install -q -U huggingface_hub hf_xet
from huggingface_hub import hf_hub_download

REPO = "unsloth/Muse-Glimmer-30B-GGUF"
MODEL_DIR = "/kaggle/temp/glimmer"
model_path = hf_hub_download(REPO, "Muse-Glimmer-30B-UD-Q4_K_XL.gguf", local_dir=MODEL_DIR)
vision_path = hf_hub_download(REPO, "mmproj-Muse-Glimmer-30B-Q8_0.gguf", local_dir=MODEL_DIR)
print(model_path)
print(vision_path)

## 4. Start the server, loaded across both GPUs

In [ ]:
import secrets
import subprocess
import time
import urllib.request

MODEL_NAME = "muse-glimmer-30b-ud-q4"
API_KEY = secrets.token_urlsafe(24)
SERVER_LOG = "/kaggle/temp/server.log"

server = subprocess.Popen(
    [
        "/kaggle/temp/llama.cpp/build/bin/llama-server",
        "--model", model_path,
        "--mmproj", vision_path,
        "-ngl", "99",
        "--tensor-split", "1,1",
        "--jinja",
        "--parallel", "3",
        "--ctx-size", "24576",
        "--alias", MODEL_NAME,
        "--api-key", API_KEY,
        "--host", "127.0.0.1",
        "--port", "8080",
    ],
    stdout=open(SERVER_LOG, "w"),
    stderr=subprocess.STDOUT,
)

ready = False
for _ in range(120):
    if server.poll() is not None:
        break
    try:
        ready = urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=3).status == 200
    except OSError:
        ready = False  # Still loading: the health check fails until the model is in memory.
    if ready:
        break
    time.sleep(5)

print("Server ready" if ready else "THE SERVER DID NOT START. Copy the lines below into the chat:")
if not ready:
    !tail -30 /kaggle/temp/server.log

## 5. Open the private link, and copy the three lines it prints

In [ ]:
import re

TUNNEL_LOG = "/kaggle/temp/tunnel.log"
!wget -q -O /kaggle/temp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /kaggle/temp/cloudflared

tunnel = subprocess.Popen(
    ["/kaggle/temp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8080", "--no-autoupdate"],
    stdout=open(TUNNEL_LOG, "w"),
    stderr=subprocess.STDOUT,
)

url = None
for _ in range(60):
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUNNEL_LOG).read())
    if found:
        url = found.group()
        break
    time.sleep(2)

if url:
    print("Replace the three LLM_ lines in your .env with these:\n")
    print(f"LLM_BASE_URL={url}/v1")
    print(f"LLM_MODEL={MODEL_NAME}")
    print(f"LLM_API_KEY={API_KEY}")
else:
    print("THE LINK DID NOT OPEN. Copy the lines below into the chat:")
    !tail -20 /kaggle/temp/tunnel.log

## 6. Keep running

This cell runs until you stop the session. Leave this tab open while your computer sends work.

In [ ]:
while True:
    state = "server ok" if server.poll() is None else "SERVER STOPPED"
    state += ", link ok" if tunnel.poll() is None else ", LINK STOPPED"
    used = subprocess.run(
        ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout.replace("\n", " | ")
    print(time.strftime("%H:%M:%S"), state, "| GPUs:", used, flush=True)
    time.sleep(300)